In [ ]:
import os
import json
import sys
import argparse
import datetime
import pandas as pd
import time
import traceback
from trade_core import InitCore
# from exchange_plugin.common_info import InitCommonInfo
# from monitor_engine.kimp_core_monitor import InitKimpCoreMonitor
# from etc.db_handler.create_schema_tables import InitDBClient
# from trigger_engine.snatcher import InitSnatcher
from etc.register_monitor_msg import RegisterMonitorMsg
from etc.redis_connector.redis_connector import InitRedis
import _pickle as pickle

# for jupyter notebook
import nest_asyncio
nest_asyncio.apply()

In [ ]:
# redis_cli = InitRedis()

In [ ]:
logging_dir = "/home/trade_core/"
with open("/home/trade_core/trade_core_config.json") as f:
    config = json.load(f)
proc_n = 1
# node = config['node']
node = 'trade_core_dev'
monitor_bot_name = config['monitor_setting']['monitor_bot']
monitor_bot_token = config['telegram_bot_setting'][monitor_bot_name]
monitor_bot_api_url = config['monitor_setting']['monitor_bot_api_url']
admin_id = config['telegram_admin_id']['charlie1155']
register_monitor_msg = RegisterMonitorMsg(monitor_bot_token, monitor_bot_api_url, admin_id, logging_dir)
# Read api keys
exchange_api_key_dict = config['exchange_api_key']

# db dict
db_dict = config['database_setting'][config['node_settings'][node]['db_settings']]

my_okx_demo_api_key = "bbb8a09a-6ea2-4686-add5-1095c918b7f4"
my_okx_demo_secret_key = "FBEAD86057449BAEC3FFA8A80CE76E4F"
my_okx_demo_passphrase = "121431Rn!!"

In [ ]:
core = InitCore(logging_dir, proc_n, node, admin_id, register_monitor_msg, exchange_api_key_dict, db_dict)

In [ ]:
conn = core.trigger.postgres_client.pool.getconn()

In [ ]:
curr = conn.cursor()

In [ ]:
core.trigger.trade_df_dict['UPBIT_SPOT/KRW:BINANCE_USD_M/USDT']

In [ ]:
target_market_code = "BITHUMB_SPOT/KRW"
origin_market_code = "OKX_USD_M/USDT"
core.get_premium_df(target_market_code, origin_market_code)

In [ ]:
from exchange_plugin.binance_plug import UserBinanceAdaptor

In [ ]:
core.trigger.trade_df_dict.get('UPBIT_SPOT/KRW:BINANCE_USD_M/USDT')

In [ ]:
core.trigger.free_trade_df_dict.get('UPBIT_SPOT/KRW:BINANCE_USD_M/USDT')

In [ ]:
user_binance_adaptor = UserBinanceAdaptor(admin_id, node, db_dict, core.trigger.free_trade_df_dict, 'UPBIT_SPOT/KRW:BINANCE_USD_M/USDT')

In [ ]:
user_binance_adaptor.start_user_socket_stream(market_type="USD_M", counterpart_liquidation_callback=lambda x: x, counterpart_margin_call_callback=lambda x: x)

In [ ]:
user_binance_adaptor.user_stream_monitoring_list[0][1].is_alive()

In [ ]:
conn = trigger.postgres_client.pool.getconn()
curr = conn.cursor(cursor_factory=extras.RealDictCursor)

sql = """
SELECT *
FROM user_info
INNER JOIN exchange_config ON user_info.user_uuid = exchange_config.user_uuid;
"""

sql = """
SELECT * FROM user_info
"""


start = time.time()
curr.execute(sql)
result = curr.fetchall()
print(time.time() - start)
# temp_df = pd.DataFrame(result)

trigger.postgres_client.pool.putconn(conn)


In [ ]:
result[0]

In [ ]:
# Add addcoin manually

from uuid import uuid4

conn = trigger.postgres_client.pool.getconn()
curr = conn.cursor(cursor_factory=extras.RealDictCursor)

sql = """
INSERT INTO trade (user_uuid, last_updated_datetime, uuid, base_asset, usdt_conversion, target_market_code, origin_market_code, low, high) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s);
"""
user_uuid = "test_uuid"
last_updated_datetime = datetime.datetime.utcnow()
uuid = uuid4().hex
base_asset = "BTC"
usdt_conversion = False
target_market_code = "UPBIT_SPOT/KRW"
origin_market_code = "BINANCE_USD_M/USDT"
low = 5
high = 6

curr.execute(sql, (user_uuid, last_updated_datetime, uuid, base_asset, usdt_conversion, target_market_code, origin_market_code, low, high))
conn.commit()

trigger.postgres_client.pool.putconn(conn)

In [ ]:
conn.rollback()
trigger.postgres_client.pool.putconn(conn)